# 01 · Validación, limpieza, EDA y entrega de base para modelado

Este notebook prepara la base de datos inicial del proyecto **BiciMAD Predictor**.

El objetivo es validar las fuentes de datos, limpiar la información necesaria, integrar variables temporales y meteorológicas, y realizar un análisis exploratorio que ayude a entender los principales riesgos operativos de las estaciones de BiciMAD.

Este notebook también deja documentado qué archivo debe utilizarse como punto de partida para la fase de preprocesamiento y modelado.

## Qué hace este notebook

- Identifica las fuentes de datos utilizadas.
- Explica para qué sirve cada dataset.
- Limpia y valida datos históricos de BiciMAD.
- Integra calendario laboral de Madrid.
- Integra meteorología histórica de AEMET.
- Crea una base enriquecida intermedia.
- Define reglas de limpieza para el análisis exploratorio.
- Analiza riesgos de estaciones casi vacías y casi llenas.
- Deja claro qué base debe usar el siguiente notebook.

## Qué no hace este notebook

- No entrena modelos.
- No define el modelo final.
- No calcula métricas predictivas.
- No realiza división train/test.
- No aplica codificación ni escalado final de variables.
- No genera el dataset final de Machine Learning en `data/processed`.

La base generada aquí será el punto de partida para que la fase de preprocesamiento y modelado tome sus propias decisiones.

## 1. Resumen rápido para el equipo de modelado

La base principal que este notebook deja preparada para las siguientes fases es:

`data/interim/bicimad/station_status_history_2022_enriched_base.csv`

Esta base contiene registros históricos de estaciones de BiciMAD durante 2022, enriquecidos con calendario laboral de Madrid y meteorología diaria observada de AEMET.

Cada fila representa el estado de una estación concreta de BiciMAD en un momento concreto del tiempo.

Dicho de forma sencilla:

> Cada fila responde a la pregunta:  
> **¿Cómo estaba una estación concreta de BiciMAD en una fecha y hora concretas?**

Esta base todavía **no es el dataset final de Machine Learning**. No contiene una variable objetivo definitiva, no tiene división train/test y no tiene codificación ni escalado final de variables.

El siguiente notebook deberá partir de esta base para definir el problema predictivo concreto, crear el target, seleccionar variables, dividir los datos temporalmente y entrenar modelos.

## 2. Explicación sencilla del problema y del dataset principal

BiciMAD es un sistema de bicicleta pública. En este tipo de sistema pueden aparecer dos problemas operativos principales:

1. Una estación puede quedarse con muy pocas bicicletas disponibles.
2. Una estación puede quedarse con muy pocos anclajes libres para devolver bicicletas.

Estos problemas no ocurren igual en todas las estaciones ni en todas las horas. Algunas estaciones pueden quedarse casi vacías en determinadas franjas horarias, mientras que otras pueden saturarse y dificultar la devolución de bicicletas.

Por eso, el proyecto necesita una base histórica que permita estudiar patrones por:

- estación,
- fecha,
- hora,
- tipo de día,
- condiciones meteorológicas,
- disponibilidad de bicicletas,
- disponibilidad de anclajes.

La base principal del proyecto se construye a partir del histórico de estado de estaciones de BiciMAD de 2022. A ese histórico se le añaden variables de calendario y meteorología para que el análisis y el futuro modelo tengan más contexto.

## 3. Mapa general del flujo de datos

El camino de los datos en este notebook sigue esta lógica:

```text
Datos originales
│
├── Histórico de estaciones BiciMAD 2022
│   └── Limpieza
│       └── station_status_history_2022_clean.csv
│
├── Calendario laboral de Madrid
│   └── Limpieza
│       └── madrid_working_calendar_clean.csv
│
├── Clima diario AEMET 2022
│   └── Limpieza y agregación
│       └── aemet_daily_weather_madrid_2022_join_ready.csv
│
└── Unión por fecha
    └── station_status_history_2022_enriched_base.csv
        └── Entrada recomendada para preprocesamiento y modelado
````

No todos los datasets descargados se usan para entrenar un modelo. Algunos sirven para enriquecer el histórico, otros para contexto de negocio y otros para una posible demo futura.

El dataset que conecta directamente con la fase de modelado es la base enriquecida histórica de estaciones:

`data/interim/bicimad/station_status_history_2022_enriched_base.csv`


## 4. Inventario de fuentes originales

Antes de limpiar o transformar datos, es importante saber de dónde viene cada fuente y para qué sirve.

Esta sección crea un inventario de datasets del proyecto. La tabla distingue entre:

- fuentes principales para análisis y modelado,
- fuentes auxiliares para enriquecer datos,
- fuentes útiles para contexto, visualización o demo futura.

El objetivo es evitar una confusión habitual: pensar que todos los datos descargados se usan directamente para entrenar el modelo. En este proyecto no es así.

In [1]:
from dataclasses import dataclass
from pathlib import Path
import pandas as pd
from IPython.display import display


@dataclass
class DataSource:
    """Representa una fuente de datos utilizada en el proyecto."""

    name: str
    source_type: str
    origin: str
    granularity: str
    project_use: str
    raw_file: str
    interim_file: str
    modeling_role: str


class DataSourceRegistry:
    """Inventario de fuentes de datos del proyecto."""

    def __init__(self, sources: list[DataSource]):
        self.sources = sources

    def to_dataframe(self) -> pd.DataFrame:
        """Devuelve el inventario en formato tabla."""
        return pd.DataFrame(
            [
                {
                    "Dataset": source.name,
                    "Tipo": source.source_type,
                    "Origen": source.origin,
                    "Granularidad": source.granularity,
                    "Uso en el proyecto": source.project_use,
                    "Archivo raw": source.raw_file,
                    "Archivo intermedio": source.interim_file,
                    "Papel para modelado": source.modeling_role,
                }
                for source in self.sources
            ]
        )


data_sources = [
    DataSource(
        name="Histórico de estaciones BiciMAD 2022",
        source_type="Fuente principal",
        origin="EMT Madrid / BiciMAD",
        granularity="Estación + momento temporal",
        project_use="Base principal para analizar disponibilidad y saturación por estación.",
        raw_file="data/raw/bicimad/bicimad_station_status_history_2022_raw.zip",
        interim_file="data/interim/bicimad/station_status_history_2022_clean.csv",
        modeling_role="Fuente base para construir el dataset de modelado.",
    ),
    DataSource(
        name="Calendario laboral de Madrid",
        source_type="Fuente auxiliar",
        origin="Portal de datos abiertos del Ayuntamiento de Madrid",
        granularity="Día",
        project_use="Añadir contexto de día laborable, sábado, domingo o festivo.",
        raw_file="data/raw/calendar/madrid_working_calendar_raw.csv",
        interim_file="data/interim/calendar/madrid_working_calendar_clean.csv",
        modeling_role="Enriquece la base principal con variables temporales.",
    ),
    DataSource(
        name="Climatología diaria AEMET 2022",
        source_type="Fuente auxiliar",
        origin="AEMET OpenData",
        granularity="Día + estación meteorológica",
        project_use="Añadir temperatura, precipitación y humedad observada al histórico 2022.",
        raw_file="data/raw/weather/aemet_daily_climate_selected_stations_2022_raw.json",
        interim_file="data/interim/weather/aemet_daily_weather_madrid_2022_join_ready.csv",
        modeling_role="Enriquece la base principal con variables meteorológicas históricas.",
    ),
    DataSource(
        name="GBFS estaciones actuales BiciMAD",
        source_type="Fuente de referencia actual",
        origin="GBFS BiciMAD",
        granularity="Estación actual",
        project_use="Referencia actual de estaciones, coordenadas, capacidad y atributos.",
        raw_file="data/raw/bicimad/gbfs_station_information_raw.json",
        interim_file="data/interim/bicimad/stations_clean.csv",
        modeling_role="No es la base de entrenamiento histórica; útil para demo, mapa o referencia actual.",
    ),
    DataSource(
        name="GBFS estado actual BiciMAD",
        source_type="Fuente de referencia actual",
        origin="GBFS BiciMAD",
        granularity="Foto actual de estación",
        project_use="Consultar disponibilidad actual de bicicletas y anclajes.",
        raw_file="data/raw/bicimad/gbfs_station_status_snapshot_raw.json",
        interim_file="data/interim/bicimad/station_status_snapshot_clean.csv",
        modeling_role="No se usa para entrenar el histórico 2022; útil para demo o comparación futura.",
    ),
    DataSource(
        name="Histórico diario de viajes BiciMAD",
        source_type="Fuente de contexto",
        origin="EMT Madrid / BiciMAD",
        granularity="Día",
        project_use="Entender la evolución general del uso de BiciMAD.",
        raw_file="data/raw/bicimad/bicimad_daily_trips_raw.csv",
        interim_file="data/interim/bicimad/bicimad_daily_trips_clean.csv",
        modeling_role="Contexto de negocio; no sustituye al histórico por estación.",
    ),
    DataSource(
        name="Inventario de estaciones AEMET",
        source_type="Fuente auxiliar",
        origin="AEMET OpenData",
        granularity="Estación meteorológica",
        project_use="Seleccionar estaciones meteorológicas cercanas y fiables para Madrid.",
        raw_file="data/raw/weather/aemet_stations_inventory_raw.json",
        interim_file="data/interim/weather/aemet_stations_inventory_clean.csv",
        modeling_role="Apoya la preparación de variables meteorológicas históricas.",
    ),
    DataSource(
        name="Predicción horaria AEMET Madrid",
        source_type="Fuente para demo futura",
        origin="AEMET OpenData",
        granularity="Hora futura",
        project_use="Demostrar cómo una app futura podría consumir meteorología prevista.",
        raw_file="data/raw/weather/aemet_forecast_hourly_madrid_raw.json",
        interim_file="data/interim/weather/aemet_forecast_hourly_madrid_clean.csv",
        modeling_role="No se usa para entrenar el histórico 2022; sirve para demo o integración futura.",
    ),
]

registry = DataSourceRegistry(data_sources)
data_sources_df = registry.to_dataframe()

display(data_sources_df)

,Dataset,Tipo,Origen,Granularidad,Uso en el proyecto,Archivo raw,Archivo intermedio,Papel para modelado
0,Histórico de estaciones BiciMAD 2022,Fuente principal,EMT Madrid / BiciMAD,Estación + momento temporal,Base principal para analizar disponibilidad y ...,data/raw/bicimad/bicimad_station_status_histor...,data/interim/bicimad/station_status_history_20...,Fuente base para construir el dataset de model...
1,Calendario laboral de Madrid,Fuente auxiliar,Portal de datos abiertos del Ayuntamiento de M...,Día,"Añadir contexto de día laborable, sábado, domi...",data/raw/calendar/madrid_working_calendar_raw.csv,data/interim/calendar/madrid_working_calendar_...,Enriquece la base principal con variables temp...
2,Climatología diaria AEMET 2022,Fuente auxiliar,AEMET OpenData,Día + estación meteorológica,"Añadir temperatura, precipitación y humedad ob...",data/raw/weather/aemet_daily_climate_selected_...,data/interim/weather/aemet_daily_weather_madri...,Enriquece la base principal con variables mete...
3,GBFS estaciones actuales BiciMAD,Fuente de referencia actual,GBFS BiciMAD,Estación actual,"Referencia actual de estaciones, coordenadas, ...",data/raw/bicimad/gbfs_station_information_raw....,data/interim/bicimad/stations_clean.csv,No es la base de entrenamiento histórica; útil...
4,GBFS estado actual BiciMAD,Fuente de referencia actual,GBFS BiciMAD,Foto actual de estación,Consultar disponibilidad actual de bicicletas ...,data/raw/bicimad/gbfs_station_status_snapshot_...,data/interim/bicimad/station_status_snapshot_c...,No se usa para entrenar el histórico 2022; úti...
5,Histórico diario de viajes BiciMAD,Fuente de contexto,EMT Madrid / BiciMAD,Día,Entender la evolución general del uso de BiciMAD.,data/raw/bicimad/bicimad_daily_trips_raw.csv,data/interim/bicimad/bicimad_daily_trips_clean...,Contexto de negocio; no sustituye al histórico...
6,Inventario de estaciones AEMET,Fuente auxiliar,AEMET OpenData,Estación meteorológica,Seleccionar estaciones meteorológicas cercanas...,data/raw/weather/aemet_stations_inventory_raw....,data/interim/weather/aemet_stations_inventory_...,Apoya la preparación de variables meteorológic...
7,Predicción horaria AEMET Madrid,Fuente para demo futura,AEMET OpenData,Hora futura,Demostrar cómo una app futura podría consumir ...,data/raw/weather/aemet_forecast_hourly_madrid_...,data/interim/weather/aemet_forecast_hourly_mad...,No se usa para entrenar el histórico 2022; sir...


## 5. Qué papel cumple cada dataset

El proyecto utiliza varios datasets, pero no todos tienen el mismo papel.

La fuente central es el histórico de estado de estaciones de BiciMAD 2022. Esta fuente indica cuántas bicicletas y anclajes había disponibles en cada estación a lo largo del tiempo.

El calendario laboral y la meteorología diaria no sustituyen a BiciMAD, sino que lo enriquecen. Sirven para añadir contexto: no es lo mismo analizar una estación en un día laborable que en un festivo, ni en un día frío que en un día templado.

Las fuentes GBFS actuales representan una fotografía reciente de la red. Son útiles para una demo, un mapa o una aplicación futura, pero no deben mezclarse automáticamente con el histórico de 2022 sin un trabajo adicional de compatibilidad.

La predicción horaria de AEMET es útil para una posible app futura, porque permite consultar el tiempo previsto. Sin embargo, no se usa para entrenar el histórico de 2022, ya que el entrenamiento necesita variables históricas observadas.

En resumen:

- **Dataset principal para el siguiente paso:** histórico enriquecido de estaciones BiciMAD 2022.
- **Datasets auxiliares:** calendario laboral y clima diario observado.
- **Datasets de contexto o demo:** GBFS actual, viajes diarios y predicción horaria.

## 6. Dataset entregado para la fase de preprocesamiento y modelado

El archivo que este notebook deja preparado como entrada recomendada para el siguiente notebook es:

`data/interim/bicimad/station_status_history_2022_enriched_base.csv`

Este archivo se guarda en `data/interim` porque todavía es una base intermedia. Está limpia, validada y enriquecida, pero no contiene las decisiones finales de Machine Learning.

La fase de preprocesamiento y modelado deberá decidir después:

1. Qué variable se quiere predecir.
2. Si el problema será de regresión o clasificación.
3. Qué horizonte temporal se quiere anticipar.
4. Cómo evitar fuga de información.
5. Cómo dividir entrenamiento y prueba respetando el orden temporal.
6. Qué variables se codifican, escalan o eliminan.
7. Qué hacer con registros no disponibles o restringidos.

Este notebook no toma esas decisiones, pero deja la base preparada y documentada para que puedan tomarse correctamente.

In [2]:
modeling_handoff = pd.DataFrame(
    [
        {
            "Elemento": "Archivo base recomendado",
            "Valor": "data/interim/bicimad/station_status_history_2022_enriched_base.csv",
        },
        {
            "Elemento": "Granularidad",
            "Valor": "Estación + momento temporal",
        },
        {
            "Elemento": "Identificador principal",
            "Valor": "station_id_historical",
        },
        {
            "Elemento": "Fecha/hora principal",
            "Valor": "snapshot_datetime",
        },
        {
            "Elemento": "Variables temporales",
            "Valor": "date, snapshot_hour, snapshot_day_of_week, snapshot_month",
        },
        {
            "Elemento": "Variables de calendario",
            "Valor": "day_type, is_working_day, is_weekend, is_holiday",
        },
        {
            "Elemento": "Variables meteorológicas",
            "Valor": "temperatura, precipitación, humedad",
        },
        {
            "Elemento": "Variables operativas",
            "Valor": "num_bikes_available, num_docks_available, occupation_ratio",
        },
        {
            "Elemento": "Ubicación del archivo",
            "Valor": "data/interim, no data/processed",
        },
        {
            "Elemento": "Responsabilidad del siguiente notebook",
            "Valor": "crear target, preprocesar variables, dividir datos y entrenar modelos",
        },
    ]
)

display(modeling_handoff)

,Elemento,Valor
0,Archivo base recomendado,data/interim/bicimad/station_status_history_20...
1,Granularidad,Estación + momento temporal
2,Identificador principal,station_id_historical
3,Fecha/hora principal,snapshot_datetime
4,Variables temporales,"date, snapshot_hour, snapshot_day_of_week, sna..."
5,Variables de calendario,"day_type, is_working_day, is_weekend, is_holiday"
6,Variables meteorológicas,"temperatura, precipitación, humedad"
7,Variables operativas,"num_bikes_available, num_docks_available, occu..."
8,Ubicación del archivo,"data/interim, no data/processed"
9,Responsabilidad del siguiente notebook,"crear target, preprocesar variables, dividir d..."
